In [4]:
MAX_VOCAB_SIZE = 20000   # or any number > max word index
MAX_SEQ_LEN = 256     # or 500, just be consistent


In [5]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split

import nltk
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

from pathlib import Path
import joblib

import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping

nltk.download('punkt_tab', quiet=True)
nltk.download('stopwords', quiet=True)
nltk.download('wordnet', quiet=True)

print("TensorFlow:", tf.__version__)


TensorFlow: 2.12.0


In [6]:
# Use the same raw or processed file as in Colab
DATA_PATH = Path("data/raw/fake_job_postings.csv")  # adjust if needed
df = pd.read_csv(DATA_PATH)

print("Dataset:", df.shape)
print("Fraud rate:", df["fraudulent"].mean() * 100)


Dataset: (17880, 18)
Fraud rate: 4.8434004474272925


In [7]:
def combine_text_fields(row, fields=(
    'title', 'location', 'company_profile', 'description',
    'requirements', 'benefits', 'required_experience',
    'required_education', 'industry', 'function',
)):
    parts = []
    for f in fields:
        if pd.notna(row[f]):
            parts.append(str(row[f]))
    return " ".join(parts) if parts else "unknown job"

print("Combining text fields...")
df["combined_text"] = df.apply(combine_text_fields, axis=1)
print("Done.")


Combining text fields...
Done.


In [8]:
stop_words = set(stopwords.words("english"))
lemmatizer = WordNetLemmatizer()

def preprocess_text(text):
    if pd.isna(text):
        return ""
    tokens = word_tokenize(str(text).lower())
    tokens = [t for t in tokens if t.isalpha() and t not in stop_words]
    tokens = [lemmatizer.lemmatize(t) for t in tokens]
    return " ".join(tokens)

print("Preprocessing text...")
df["text_processed"] = df["combined_text"].apply(preprocess_text)
print("Done.")


Preprocessing text...
Done.


In [9]:
X_text = df["text_processed"].values
y = df["fraudulent"].values.astype("int32")

# First: train vs temp (val+test)
X_train_text, X_temp_text, y_train, y_temp = train_test_split(
    X_text,
    y,
    test_size=0.30,        # 70% train, 30% temp
    random_state=42,
    stratify=y,
)

# Second: split temp into val and test (15% + 15%)
X_val_text, X_test_text, y_val, y_test = train_test_split(
    X_temp_text,
    y_temp,
    test_size=0.50,
    random_state=42,
    stratify=y_temp,
)

print(len(X_train_text), len(X_val_text), len(X_test_text))


12516 2682 2682


In [10]:
tokenizer = Tokenizer(num_words=MAX_VOCAB_SIZE, oov_token="<OOV>")
tokenizer.fit_on_texts(X_train_text)

X_train_seq = tokenizer.texts_to_sequences(X_train_text)
X_val_seq   = tokenizer.texts_to_sequences(X_val_text)
X_test_seq  = tokenizer.texts_to_sequences(X_test_text)

X_train_pad = pad_sequences(X_train_seq, maxlen=MAX_SEQ_LEN, padding="post", truncating="post")
X_val_pad   = pad_sequences(X_val_seq,   maxlen=MAX_SEQ_LEN, padding="post", truncating="post")
X_test_pad  = pad_sequences(X_test_seq,  maxlen=MAX_SEQ_LEN, padding="post", truncating="post")


In [11]:
from tensorflow.keras.optimizers import Adam

print("\nBuilding LSTM model...")

EMBEDDING_DIM = 64
LSTM_UNITS    = 64
DENSE_UNITS   = 32


from tensorflow.keras.layers import LSTM, Bidirectional

model = Sequential([
    Embedding(
        input_dim=MAX_VOCAB_SIZE,
        output_dim=EMBEDDING_DIM,
        input_length=MAX_SEQ_LEN,
    ),
    Bidirectional(LSTM(
        LSTM_UNITS,
        dropout=0.3,
        recurrent_dropout=0.3,
    )),
    Dropout(0.3),
    Dense(DENSE_UNITS, activation="relu"),
    Dense(1, activation="sigmoid"),
])

optimizer = Adam(learning_rate=5e-4)  # 0.0005

model.compile(
    loss="binary_crossentropy",
    optimizer=optimizer,
    metrics=["accuracy"],
)

model.summary()


Building LSTM model...
Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 embedding (Embedding)       (None, 256, 64)           1280000   
                                                                 
 bidirectional (Bidirectiona  (None, 128)              66048     
 l)                                                              
                                                                 
 dropout (Dropout)           (None, 128)               0         
                                                                 
 dense (Dense)               (None, 32)                4128      
                                                                 
 dense_1 (Dense)             (None, 1)                 33        
                                                                 
Total params: 1,350,209
Trainable params: 1,350,209
Non-trainable params: 0
______________________

In [12]:
early_stop = EarlyStopping(
    monitor="val_loss",
    patience=3,
    restore_best_weights=True,
    verbose=1,
)

history = model.fit(
    X_train_pad, y_train,
    validation_data=(X_val_pad, y_val),
    epochs=8,
    batch_size=256,   # or 256
    callbacks=[early_stop],
    verbose=1,
)



Epoch 1/8
49/49 [==============================] - 122s 2s/step - loss: 0.3827 - accuracy: 0.9463 - val_loss: 0.1922 - val_accuracy: 0.9515
Epoch 2/8
49/49 [==============================] - 130s 3s/step - loss: 0.1542 - accuracy: 0.9516 - val_loss: 0.0901 - val_accuracy: 0.9515
Epoch 3/8
49/49 [==============================] - 124s 3s/step - loss: 0.0700 - accuracy: 0.9732 - val_loss: 0.0535 - val_accuracy: 0.9866
Epoch 4/8
49/49 [==============================] - 121s 2s/step - loss: 0.0392 - accuracy: 0.9885 - val_loss: 0.0388 - val_accuracy: 0.9873
Epoch 5/8
49/49 [==============================] - 129s 3s/step - loss: 0.0188 - accuracy: 0.9934 - val_loss: 0.0341 - val_accuracy: 0.9888
Epoch 6/8
49/49 [==============================] - 132s 3s/step - loss: 0.0098 - accuracy: 0.9970 - val_loss: 0.0422 - val_accuracy: 0.9896
Epoch 7/8
49/49 [==============================] - 132s 3s/step - loss: 0.0060 - accuracy: 0.9982 - val_loss: 0.0424 - val_accuracy: 0.9896
Epoch 8/8
49/49 [===

In [13]:
val_loss, val_acc = model.evaluate(X_val_pad, y_val, verbose=0)
print(f"Validation accuracy: {val_acc:.4f}")


Validation accuracy: 0.9888


In [14]:
y_test_pred_proba = model.predict(X_test_pad).ravel()
y_test_pred = (y_test_pred_proba >= 0.5).astype(int)

from sklearn.metrics import classification_report, confusion_matrix

print(confusion_matrix(y_test, y_test_pred))
print(classification_report(y_test, y_test_pred, digits=4))

84/84 [==============================] - 3s 35ms/step
[[2533   19]
 [  24  106]]
              precision    recall  f1-score   support

           0     0.9906    0.9926    0.9916      2552
           1     0.8480    0.8154    0.8314       130

    accuracy                         0.9840      2682
   macro avg     0.9193    0.9040    0.9115      2682
weighted avg     0.9837    0.9840    0.9838      2682



In [15]:
from pathlib import Path
import joblib

MODELS_DIR = Path("models")
MODELS_DIR.mkdir(exist_ok=True)

joblib.dump(tokenizer, MODELS_DIR / "tokenizer.pkl")
model.save(MODELS_DIR / "lstm_model.h5")

print("Saved tokenizer.pkl and lstm_model.h5")


Saved tokenizer.pkl and lstm_model.h5


In [16]:
from pathlib import Path

MODELS_DIR = Path("models")
savedmodel_dir = MODELS_DIR / "lstm_savedmodel"
savedmodel_dir.mkdir(exist_ok=True)

model.save(savedmodel_dir, save_format="tf")  # Keras SavedModel
print("Saved LSTM SavedModel to", savedmodel_dir)


INFO:tensorflow:Assets written to: models\lstm_savedmodel\assets


INFO:tensorflow:Assets written to: models\lstm_savedmodel\assets


Saved LSTM SavedModel to models\lstm_savedmodel
